# L3 Analysis

L2/3-focused analysis split out from `wiring_efficiency.ipynb`.

In [ ]:
%load_ext autoreload
%autoreload 2

import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from tqdm import tqdm
from torchvision.transforms import functional as TF
from mpl_toolkits.mplot3d import Axes3D
from IPython.display import display, Image as IPImage
import gc
import matplotlib.colors as mcolors

from helpers.wiring_efficiency_utils import *
from neuralsheet import *
from helpers.map_plotting import *
from helpers.fig_plots import *
from helpers.l3_plotting import (
    run_connection_pool_model_figure,
    run_normalised_excess_sparsity_analysis,
    run_parity_lambda_summary,
    run_sparsity_summary_and_beta_validation,
)


In [ ]:
# Example usage
crop_size = 24 # Crop size (NxN)
batch_size = 32  # Number of crops to load at once
num_workers = 4  # Number of threads for data loading
root_dir = './input_stimuli'  # Path to your image folder
device = 'cuda'  # Assuming CUDA is available and desired
beta = 1 - 5e-5
loss_beta = 1e-2
R_rf = 7

dataloader = create_dataloader(root_dir, crop_size, batch_size, num_workers)

In [ ]:
results = run_parity_lambda_summary()


In [ ]:
torch.load("./parameter_search_data/search_results_micro_grid.pt", weights_only=False)['dimensionality'][2].squeeze()

In [ ]:
connection_pool_results = run_connection_pool_model_figure()


In [ ]:
results, normalised_panel_results = run_normalised_excess_sparsity_analysis()


In [ ]:
import torch

keys_to_reshape_cat = [
    "accuracy",
    "dimensionality",
    "robustness",
    "columnarity",
    "sparsity",
    "loc_sparsity",
]

f1 = "parameter_search_data/search_results_macro_grid.pt"
f2 = "parameter_search_data/search_results_macro_GRID.pt"

d1 = torch.load(f1, map_location="cpu", weights_only=False)
d2 = torch.load(f2, map_location="cpu", weights_only=False)

out = {}

def reshape_for_cat(x):
    if x.ndim < 3:
        return x
    return x.reshape(x.shape[0], x.shape[1] * x.shape[2], 1, *x.shape[3:])

for k in d1:
    if k in keys_to_reshape_cat:
        v1 = reshape_for_cat(d1[k])
        v2 = reshape_for_cat(d2[k])

        assert v1.shape[0] == v2.shape[0]
        assert v1.shape[2:] == v2.shape[2:]

        out[k] = torch.cat([v1, v2], dim=0).view(6,6,6,1,-1)
        if out[k].shape[-1]==1:
            out[k] = out[k][:,:,:,:,0]
        print(k, d1[k].shape, d2[k].shape, "->", out[k].shape)
    else:
        out[k] = d1[k]

torch.save(out, "parameter_search_data/search_results_macro_grid_joint.pt")

In [ ]:
(
    fig_fit,
    axes_fit,
    curve_dict,
    fig_val,
    axes_val,
    validation_stats,
) = run_sparsity_summary_and_beta_validation()


In [ ]:
def train_single_model(microcolumnar, p0: float, p1: float, p2: float, sheet_size: int, r_long: float):
    model = NeuralSheet(
        crop_size,
        sheet_size,
        R_rf,
        R_long=r_long,
        device=device,
        microcolumnar=microcolumnar,
        p = [0, p1, p2]
    ).to(device)

    lr = 1e-3
    network = init_nn(sheet_size, crop_size, 1)
    avg_loss = 0.0

    model.train()
    for _ in range(1):
        batch_progress = dataloader #tqdm(dataloader, leave=False, desc=f"train R_long={r_long:.2f}", disable=not sys.stdout.isatty())
        for batch in batch_progress:
            batch = batch.to(device)
            batch_responses = []
            batch_inputs = []

            for image in batch:
                image = image[0:1][None].flip(1)

                if image.mean() <= 0.15:
                    continue

                lr = max(lr * beta, 1e-4)
                model.hebbian_lr = lr * 1e2
                model.homeo_lr = lr

                #if lr < 2e-4:
                #    print('target lr reached, training complete')
                #    return model, avg_loss

                model(image, adaptation=True, layer_3=True)
                model.hebbian_step(layer_3=True)

                batch_responses.append(model.current_response_l3.clone())
                batch_inputs.append(model.current_input.clone())

            if not batch_responses:
                continue

            batch_responses = torch.cat(batch_responses, dim=0)
            batch_inputs = torch.cat(batch_inputs, dim=0)

            reco_input = network['activ'](network['model'](batch_responses))
            targets = batch_inputs

            loss, _ = nn_loss(network, targets, reco_input)
            sim = cosim(targets.detach().cpu(), reco_input.detach().cpu(), True)
            avg_loss = (1 - loss_beta) * avg_loss + loss_beta * sim

            network['optim'].zero_grad()
            loss.backward()
            network['optim'].step()

    return model, avg_loss, lr

model, avg_loss, lr = train_single_model(True, 0.04, 0.5, 0.2, 40, 12)

In [ ]:
codes = []

for e in range(1):
    
    batch_progress = tqdm(dataloader, leave=False)
    for b_idx, batch in enumerate(batch_progress):
    
        for image in batch:

            image = image[0:1][None].flip(1).cuda()
    
            if image.mean() > 0.15:
    
                model(image, adaptation=False, noise_gamma=0.0, sparsity=0., layer_3=True)
                
                codes.append(model.current_response_l3.clone().cpu())
    
        if b_idx > 150:
            
            break

codes = torch.cat(codes, dim=0)
eff_dim, comp_sampled = get_pca_dimensions(codes, 3)
print(eff_dim)

In [ ]:
def increase_lateral_influence(model):

    model.p2 = 0.015
    lr = 5e-4
    model.hebbian_lr = lr * 1e2
    model.homeo_lr = lr

    batch_progress = dataloader #tqdm(dataloader, leave=False, desc=f"train R_long={r_long:.2f}", disable=not sys.stdout.isatty())
    for b_idx, batch in enumerate(batch_progress):
        batch = batch.to(device)
    
        for image in batch:
            image = image[0:1][None].flip(1)
    
            if image.mean() <= 0.15:
                continue
    
            model(image, adaptation=True)

increase_lateral_influence(model)

In [ ]:
batch_progress = tqdm(dataloader, leave=False)
trials = 20
samples = 300

scores = np.zeros((samples, trials, trials, trials))
vals = np.linspace(1, trials, trials)
vals = [0.]
norms = [0]
c = 0
a = int(trialvar[0]//2)

for b_idx, batch in enumerate(batch_progress):

    for image in batch:
    
        img = image[0:1][None].flip(1).cuda()
        model(img, adaptation=False, layer_3=True)
        dense = model.current_response_l3.clone().cpu() + 0

        if dense[:,:,a:-a-1,a:-a-1].mean() > 0.05:
    
            for v_idx, v in enumerate(vals):

                for p_idx, p in enumerate(norms): 

                    #model.lri_norm = np.exp(-v + 1) * 0.8 + 0.2
                    #model.mri_norm = p #np.exp(-v + 1) * 0.8 + 0.2
                    #model.sre_norm = np.exp(-v + 1) # p
                    
                    for pi_idx, pi in enumerate(norms): 

                        #model.sre_norm = np.exp(-v + 1)
                        #model.lre_norm = np.exp(-v + 1) * 0.5 + 0.5
                        #model.needs_update=True
                        #o = 10
                        #ps = (v + o - 1) / o
                        model(img, adaptation=False, sparsity=v, loc_sparsity=0., noise_gamma=0.05, layer_3=True)
                        #plt.imshow(model.long_range_exc[1000,0].cpu())
                        #plt.show()
                        sparse = model.current_response_l3.clone().cpu() + 0
                        scores[c, v_idx, p_idx, pi_idx] = cosim(dense[:,:,a:-a-1,a:-a-1], sparse[:,:,a:-a-1,a:-a-1])   
                        break
                    break

            c += 1
            #print('image no. ' + str(c))

        if c == samples:
            break
            

    if c == samples:
        break

scores[scores>0.1].mean(0).max()

In [ ]:
get_masses_and_spreads(model.lateral_correlations_exc_l3)[0].mean()

In [ ]:
umap_results = collect_and_plot_grating_umap_3d(
    model,
    crop_size,
    device=device,
    n=10000,
    wavelength=6,
    include_noise=True,
    fit_umap_on_noise=False,
    use_l3=True,
)

code_tracker = list(umap_results["clean_codes"].split(1, dim=0))
angles = umap_results["angles"]
clean = (umap_results["clean_codes"], torch.tensor(angles))
noisy = (umap_results.get("noisy_codes"), torch.tensor(angles))


In [ ]:
# Looping over the DataLoader

trials = 15
trialvar = np.sqrt(np.linspace(9**2, 20**2, trials))
sizesvar = np.round(np.sqrt(np.linspace(400, 2500, 3))).astype(int)
sizesvar = [30]
sizes = len(sizesvar)
trials = len(trialvar)
epochs = 100000
baseline = 0 #0.64

n_map_frames = 30
it_gap = 10
map_tracker = torch.zeros((n_map_frames, 1, sizesvar[-1], sizesvar[-1]))

gc.collect()


for s in range(sizes):
    
    for t in range(trials):
        
        KEEP = False
        
        if not KEEP:
            model = NeuralSheet(crop_size, sizesvar[s], R_rf, R_long=trialvar[t], device=device, microcolumnar=False, p=[0, 0.5, 0.2]).to(device)
            lr = 1e-3
    
            network = init_nn(sizesvar[s], crop_size, 1)
            avg_loss = 0

            code_tracker = []
        
        for e in range(epochs):

            batch_progress = tqdm(dataloader, leave=False)
            for b_idx, batch in enumerate(batch_progress):

                if (b_idx%it_gap)==0:
                    map_tracker = map_tracker.roll(dims=0, shifts=-1)
                    last_map = detect_orientation_map_from_aff_weights(model.afferent_weights)['pref'].view(sizesvar[0], sizesvar[0]) / 180 * np.pi
                    map_tracker[-1,0] = last_map.view(model.sheet_size, model.sheet_size)

                batch_responses = []
                batch_inputs = []
                batch = batch.to('cuda')  # Transfer the entire batch to GPU

                for image in batch:

                    image = image[0:1][None].flip(1)

                    if image.mean()>0.15:
                        
                        limit = 3e-4
                        lr *= beta
                        lr = lr if lr>limit else limit
                        model.hebbian_lr = lr * 1e2
                        model.homeo_lr = lr

                        model(image, adaptation=True, noise_gamma=0.0, sparsity=0., layer_3=True)
                        model.hebbian_step(layer_3=True)
                        
                        batch_responses.append(model.current_response_l3.clone().detach())
                        batch_inputs.append(model.current_input.clone())
                        code_tracker.append(model.current_response_l3.clone())

                if len(batch_responses):
                    batch_responses = torch.cat(batch_responses, dim=0)
                    batch_inputs = torch.cat(batch_inputs, dim=0)

                    reco_input = network['activ'](network['model'](batch_responses)) 
                    targets = batch_inputs
                    
                    loss, loss_std = nn_loss(network, targets, reco_input)
                    
                    sim = cosim(targets.detach().cpu(), reco_input.detach().cpu(), True) - baseline
                    sim /= 1 - baseline
                                    
                    avg_loss = (1-loss_beta)*avg_loss + loss_beta*sim
                    
                    network['optim'].zero_grad()
                    loss.backward()
                    network['optim'].step()
    
                    mean_activation = model.mean_activations_l3.mean()
                    mean_std = model.mean_activations_l3.std() / model.homeo_target    
                    
                    batch_progress.set_description('M:{:.3f} STD:{:.3f} BCE:{:.3f} LR:{:.5f} SP:{:.3f} D:{:.3f} A:{:.3f}'.format(
                        mean_activation, 
                        mean_std, 
                        avg_loss,
                        lr,
                        model.aff_strength_l3.mean(),
                        model.gains_l3.mean(),
                        model.delta_mag.mean()
                    ))

In [ ]:
model.iterations = 100
model.response_tracker_l3 = torch.zeros(
    model.iterations,
    1,
    model.sheet_size,
    model.sheet_size,
    device=model.device,
)


In [ ]:
cosim(model.lateral_correlations, model.lateral_correlations_l3)

In [ ]:
%matplotlib inline
#model.update_interactions(1,1,1)
random_sample = random.randint(0, model.afferent_weights.shape[0] - 1)
random_batch = random.randint(0, batch_size - 1)
    
#%lprun -f model.forward model.forward(batch[random_batch, 0:1][None].flip(1),rf_grids)
model.forward(batch[random_batch, 0:1][None].flip(1).cuda(), noise_gamma=0.)

model.current_response.max()
array = model.response_tracker.cpu()[:,0]
array[:,0,0] = 1
anim = animate(array, model.iterations)

show_map(model, network, random_sample)
anim

In [ ]:
from matplotlib.colors import LinearSegmentedColormap
def imshow_and_save(tensor, save_path, color="red", vmin=0, vmax=None):
    if hasattr(tensor, "detach"):
        tensor = tensor.detach().cpu().numpy()

    img = tensor[0]
    if vmax is None:
        vmax = img.max()

    cmap = LinearSegmentedColormap.from_list("white_to_color", [(1, 1, 1), color])
    plt.figure(figsize=(8,8))
    plt.imshow(img, cmap=cmap, vmin=vmin, vmax=vmax)
    plt.axis("off")
    plt.tight_layout(pad=0)
    plt.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.show()

imshow_and_save(model.global_exc_l3[1830], './figures/root_exports/sp_g_exc.png', color='red')


In [ ]:
%matplotlib inline
#model.update_interactions(1,1,1)
#random_sample = random.randint(0, model.afferent_weights_l3.shape[0] - 1)
#random_batch = random.randint(0, batch.shape[0] - 1)
#print(batch.shape)

#%lprun -f model.forward model.forward(batch[random_batch, 0:1][None].flip(1),rf_grids)
model.forward(batch[random_batch, 0:1][None].flip(1).cuda(), noise_gamma=0.0, sparsity=0.)

array = model.response_tracker_l3.cpu()[:,0]
array[:,0,0] = 1
anim = animate(array, model.iterations)

show_map_l3(model, network, random_sample)
anim